# GVHMR on Colab — Track A fork

Adapted from [zju3dv/GVHMR](https://github.com/zju3dv/GVHMR) `tools/demo/colab_demo.ipynb`.

The official notebook pins `torch==2.3.0+cu121` and a py3.10 pytorch3d wheel. This copy keeps Colab's torch, skips DPVO, and exports `gvhmr_smpl.npz` for `scripts/ingest_smpl.py`.

Paper: https://arxiv.org/abs/2409.06662

**Runtime → T4 GPU.** If Colab asks to restart after pip, click **Cancel**.


## 1. GPU check


In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0], "torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch runtime to T4 GPU"


## 2. Install GVHMR (do not replace Colab torch, no DPVO)


In [ ]:
from pathlib import Path
%cd /content
if not Path("/content/GVHMR").exists():
    !git clone --depth 1 https://github.com/zju3dv/GVHMR /content/GVHMR
%cd /content/GVHMR
req = Path("requirements.txt").read_text().splitlines()
skip = ("--extra-index-url", "torch==", "torchvision==", "pytorch3d", "numpy==")
keep = []
for line in req:
    s = line.strip()
    if not s or s.startswith("#"):
        continue
    if any(s.startswith(p) for p in skip):
        print("skip pin:", s)
        continue
    keep.append(s)
Path("/tmp/gvhmr_req_colab.txt").write_text("\n".join(keep) + "\n")
%pip install -q -r /tmp/gvhmr_req_colab.txt
%pip install -q -e .


In [ ]:
try:
    import pytorch3d
    print("pytorch3d ok", getattr(pytorch3d, "__version__", ""))
except Exception:
    print("pytorch3d missing; preview video may fail; .pt export still works")
    try:
        %pip install -q pytorch3d
    except Exception as exc:
        print(exc)


## 3. Checkpoints

Mirrors used by the official notebook (camenduru/GVHMR, camenduru/SMPLer-X).
SMPL/SMPL-X are MPI-licensed: https://smpl.is.tue.mpg.de / https://smpl-x.is.tue.mpg.de


In [ ]:
from pathlib import Path
%cd /content/GVHMR
!mkdir -p inputs outputs
!apt-get install -y -qq aria2 >/dev/null

def fetch(url, dest_dir, name):
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    out = Path(dest_dir) / name
    if out.exists() and out.stat().st_size > 1000:
        print("have", out)
        return
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest_dir}" -o "{name}"

S = "https://huggingface.co/camenduru/SMPLer-X/resolve/main"
G = "https://huggingface.co/camenduru/GVHMR/resolve/main"
fetch(f"{S}/SMPL_NEUTRAL.pkl", "inputs/checkpoints/body_models/smpl", "SMPL_NEUTRAL.pkl")
fetch(f"{S}/SMPLX_NEUTRAL.npz", "inputs/checkpoints/body_models/smplx", "SMPLX_NEUTRAL.npz")
fetch(f"{G}/gvhmr/gvhmr_siga24_release.ckpt", "inputs/checkpoints/gvhmr", "gvhmr_siga24_release.ckpt")
fetch(f"{G}/hmr2/epoch%3D10-step%3D25000.ckpt", "inputs/checkpoints/hmr2", "epoch=10-step=25000.ckpt")
fetch(f"{G}/vitpose/vitpose-h-multi-coco.pth", "inputs/checkpoints/vitpose", "vitpose-h-multi-coco.pth")
fetch(f"{G}/yolo/yolov8x.pt", "inputs/checkpoints/yolo", "yolov8x.pt")
!find inputs/checkpoints -type f | sort


## 4. Optional smoke test (tennis clip, static cam). Skip to save quota.


In [ ]:
!python /content/GVHMR/tools/demo/demo.py --video=/content/GVHMR/docs/example_video/tennis.mp4 -s


## 5. Upload your exercise clip and run with -s


In [ ]:
from google.colab import files
from pathlib import Path
STATIC_CAM = True
d = Path("/content/GVHMR/inputs/demo")
d.mkdir(parents=True, exist_ok=True)
print("Choose your exercise mp4")
uploaded = files.upload()
name = next(iter(uploaded))
dst = d / name.replace(" ", "_")
dst.write_bytes(uploaded[name])
print("saved", dst, dst.stat().st_size)


In [ ]:
from pathlib import Path
clips = [p for p in Path("/content/GVHMR/inputs/demo").iterdir() if p.suffix.lower() in {".mp4", ".mov", ".webm", ".mkv"}]
assert clips, "upload a video first"
video = clips[-1]
flag = "-s" if STATIC_CAM else ""
print("running", video, flag)
!python /content/GVHMR/tools/demo/demo.py --video={video} {flag}


## 6. Export gvhmr_smpl.npz


In [ ]:
from pathlib import Path
import torch, numpy as np
from google.colab import files

def to_np(x):
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)

pts = list(Path("/content/GVHMR/outputs").rglob("*"))
pts = [p for p in pts if p.suffix == ".pt" and ("hmr4d" in p.name.lower() or p.name.endswith("results.pt"))]
if not pts:
    pts = list(Path("/content/GVHMR/outputs").rglob("*.pt"))
print("pt files:", pts)
blob = torch.load(pts[0], map_location="cpu", weights_only=False)
print("keys", list(blob) if isinstance(blob, dict) else type(blob))
params = None
for k in ("smpl_params_global", "smpl_params_incam", "smpl_params"):
    if isinstance(blob, dict) and k in blob:
        params = blob[k]
        print("using", k)
        break
pack = {}
for dst, srcs in {
    "body_pose": ("body_pose", "pose_body"),
    "global_orient": ("global_orient", "root_orient"),
    "transl": ("transl", "trans"),
    "betas": ("betas",),
}.items():
    for s in srcs:
        if s in params:
            pack[dst] = to_np(params[s])
            break
print({k: v.shape for k, v in pack.items()})
out = Path("/content/gvhmr_smpl.npz")
np.savez_compressed(out, **pack)
files.download(str(out))


## 7. On the Mac

```bash
python scripts/ingest_smpl.py --motion output/<stem>_motion.npz --gvhmr ~/Downloads/gvhmr_smpl.npz
```

Then `docs/MAC_BLENDER.md`.
